# 05 · Merge variants with the flux product

Combines, on the shared half-hourly timestamp index:

- all six time-lag **variant subsets** (`data/02-…_subsets/`), and
- the **FP2025.3** 2024 reference (`data/04-flux-product-2025.3_subset_2024/`).

The variants share identical column names (`FN2O`, `N2O_TLAG_USED`, …), so each
source's columns get a **suffix** identifying it (e.g. `FN2O_LGR-1`,
`FN2O_FP2025.3`). The merged wide table is saved to
`data/05-merged_variants_fluxproduct/`.

## Imports

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from diive.core.io.files import load_parquet, save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Configuration

In [2]:
SUBSETDIR = Path("../data/02-eddypro_fluxes_level-1_parquet_subsets")  # variant subsets
FPDIR = Path("../data/04-flux-product-2025.3_subset_2024")            # flux product 2024
OUTDIR = Path("../data/05-merged_variants_fluxproduct")
OUTDIR.mkdir(parents=True, exist_ok=True)

FP_SUFFIX = "FP2025.3"  # suffix applied to the flux-product columns
OUTNAME = "merged_variants_fp2025.3_2024"  # output filename (without extension)

## Load and suffix the variants

Each variant's columns are suffixed with its version code (the Parquet file
stem), e.g. `FN2O` → `FN2O_LGR-1`.

In [3]:
frames = []
for f in sorted(SUBSETDIR.glob("*.parquet")):
    code = f.stem  # version code, e.g. 'LGR-1'
    df = load_parquet(filepath=str(f)).add_suffix(f"_{code}")
    print(f"{code}: {df.shape[0]} rows x {df.shape[1]} cols")
    frames.append(df)

> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet (0.098 seconds).

LGR-1: 7805 rows x 4 cols


> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-2R.parquet (0.036 seconds).

LGR-2R: 7805 rows x 4 cols


> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-3.parquet (0.061 seconds).

...

QCL-1: 9642 rows x 4 cols


> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-2R.parquet (0.048 seconds).

QCL-2R: 9642 rows x 4 cols


> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-3.parquet (0.048 seconds).

QCL-3: 20790 rows x 4 cols


## Load and suffix the flux product

In [4]:
fp_files = sorted(FPDIR.glob("*.parquet"))
assert len(fp_files) == 1, f"expected one flux-product file, found {len(fp_files)}"
fp = load_parquet(filepath=str(fp_files[0])).add_suffix(f"_{FP_SUFFIX}")
print(f"{FP_SUFFIX}: {fp.shape[0]} rows x {fp.shape[1]} cols")
frames.append(fp)

> Loaded .parquet file ..\data\04-flux-product-2025.3_subset_2024\CH-CHA_FP2025.3_subset_2024.parquet (0.023 
seconds).

FP2025.3: 17568 rows x 39 cols


## Merge

Outer join on the timestamp index so all records from every source are kept
(variants cover different periods; the flux product is 2024 only).

In [5]:
merged = pd.concat(frames, axis=1, join="outer").sort_index()
print(f"Merged: {merged.shape[0]} rows x {merged.shape[1]} cols")
print(f"Range: {merged.index.min()} -> {merged.index.max()}")
merged.head(10)

Merged: 55126 rows x 63 cols
Range: 2020-05-13 13:15:00 -> 2024-12-31 23:45:00


C:\Users\holukas\AppData\Local\Temp\ipykernel_27172\1820004119.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  merged = pd.concat(frames, axis=1, join="outer").sort_index()


,FN2O_LGR-1,FCH4_LGR-1,N2O_TLAG_USED_LGR-1,CH4_TLAG_USED_LGR-1,FN2O_LGR-2R,FCH4_LGR-2R,N2O_TLAG_USED_LGR-2R,...,.PREC_RAIN_TOT_GF1_0.5_1_MEAN3H-6_FP2025.3,TS_GF1_0.04_1_gfXG_FP2025.3,TS_GF1_0.15_1_gfXG_FP2025.3,TS_GF1_0.4_1_gfXG_FP2025.3,SWC_GF1_0.15_1_gfXG_FP2025.3,PREC_RAIN_TOT_GF1_0.5_1_FP2025.3,USTAR_FP2025.3
TIMESTAMP_MIDDLE,,,,,,,,,,,,,,,
2020-05-13 13:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 13:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 14:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 14:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 15:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 15:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 16:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 16:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-05-13 17:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Save

In [6]:
filepath = save_parquet(filename=OUTNAME, data=merged, outpath=str(OUTDIR))
print(f"Saved {filepath}")

> Saved file ..\data\05-merged_variants_fluxproduct\merged_variants_fp2025.3_2024.parquet (0.206 seconds).

Saved ..\data\05-merged_variants_fluxproduct\merged_variants_fp2025.3_2024.parquet


## Runtime

In [7]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-06-04 15:50:06
End:      2026-06-04 15:50:08
Runtime:  0:00:02.161045
